# client

> Client for interacting with the Fewsats API

In [ ]:
#| default_exp core

## Class

In [ ]:
#| export
from fastcore.utils import *
import os
import hashlib
import hmac
import httpx
import time
import json
from dataclasses import dataclass
from fastcore.basics import BasicRepr
from fastcore.utils import store_attr
from typing import List, Dict, Any
from functools import wraps


In [ ]:
#| hide 
from dotenv import load_dotenv
from fastcore.test import *

In [ ]:
#| hide
load_dotenv()

True

The `Fewsats` class handles authentication and provides the foundation for our API interactions.

In [ ]:
#| export
class Fewsats:

    WEBHOOK_VERSION = "v1"
    WEBHOOK_SIGNATURE_HEADER = "Fewsats-Signature"

    "Client for interacting with the Fewsats API"
    def __init__(self,
                 api_key: str = None, # The API key for the Fewsats account
                 base_url: str = "https://api.fewsats.com"): # The Fewsats API base URL
        self.api_key = api_key or os.environ.get("FEWSATS_API_KEY")
        if not self.api_key:
            raise ValueError("The api_key client option must be set either by passing api_key to the client or by setting the FEWSATS_API_KEY environment variable")
        self.base_url = base_url
        self._httpx_client = httpx.Client()
        self._httpx_client.headers.update({"Authorization": f"Token {self.api_key}"})

    def _request(self,
                method: str, # The HTTP method to use
                path: str, # The path to request
                timeout: int = 10, # Timeout for the request in s
                **kwargs) -> Dict[str, Any]:
        "Makes an authenticated request to Fewsats API"
        url = f"{self.base_url}/{path}"
        return  self._httpx_client.request(method, url, timeout=timeout, **kwargs)


In [ ]:
k = os.getenv("FEWSATS_API_KEY")
fs = Fewsats(api_key=k)
k = os.getenv("FEWSATS_LOCAL_API_KEY")
fs = Fewsats(api_key=k, base_url="http://localhost:8000")

test_eq(fs.api_key, k)
test_eq(fs._httpx_client.headers["Authorization"], f"Token {k}")

## Methods

### User & Billing Info

In [ ]:
#| export

@patch
def me(self: Fewsats):
    "Retrieve the user's info."
    return self._request("GET", "v0/users/me")


In [ ]:
r = fs.me()
r.status_code, r.json()

(200,
 {'name': 'Pol',
  'last_name': 'Alvarez Vecino',
  'email': 'pol@fewsats.com',
  'id': 1,
  'created_at': '2024-08-20T16:13:01.255Z',
  'webhook_secret': 'whsec_Kgsdk1xApAOSBcpXViSMNp_MMCRwfNWngoxqotZMHUw',
  'test_webhook_secret': 'whsec_Nk997GrUMG9rwEmlef7jlMCMfz2th3chUHtRBkcScIU',
  'webhooks': [{'id': 4, 'url': 'https://example.com', 'is_test': False},
   {'id': 8, 'url': 'https://example.com/webhook', 'is_test': False}],
  'test_webhooks': []})

In [ ]:
#| export

@patch
def billing_info(self: Fewsats):
    "Retrieve the user's billing info."
    return self._request("GET", "v0/users/me/billing-info")

In [ ]:
billing_info = fs.billing_info()
billing_info.status_code, billing_info.json()

(200,
 {'full_name': 'Test Name 2',
  'company_name': '',
  'vat_number': None,
  'address': 'Fake Street 123',
  'address_line2': '',
  'city': 'Crystal City',
  'state': '',
  'postal_code': '3129',
  'country': 'US',
  'phone': None})

### Balance 

In [ ]:
#| export 

@patch
def balance(self: Fewsats):
    "Retrieve the balance of the user's wallet. Amounts are always in USD cents."
    return self._request("GET", "v0/wallets")


In [ ]:
r = fs.balance()
r.status_code, r.json()

(200, [{'id': 15, 'balance': 87, 'currency': 'usd'}])

### Payment Methods

Retrieve the user's payment methods. Useful for checking which card will be used for purchases.

In [ ]:
#| export
@patch
def payment_methods(self: Fewsats) -> List[Dict[str, Any]]:
    "Retrieve the user's payment methods, raises an exception for error status codes."
    return self._request("GET", "v0/stripe/payment-methods")


In [ ]:
r = fs.payment_methods()
payment_methods = r.json()
r.status_code, payment_methods

(200,
 [{'id': 1,
   'last4': '4242',
   'brand': 'visa',
   'exp_month': 12,
   'exp_year': 2034,
   'is_default': False},
  {'id': 4,
   'last4': '4242',
   'brand': 'Visa',
   'exp_month': 12,
   'exp_year': 2034,
   'is_default': True}])

In [ ]:
assert isinstance(payment_methods, list)

### Preview a Purchase

Preview the resulting state of a purchase. Useful, for example, to check if a CC charge is needed or the purchase will use the balance.

In [ ]:
#| export

@patch
def _preview_payment(self: Fewsats,
                    amount: str): # The amount in USD cents
    "Simulates a purchase, raises an exception for error status codes."
    assert amount.isdigit()
    return self._request("POST", "v0/l402/preview/purchase/amount", json={"amount_usd": amount})


In [ ]:
r = fs._preview_payment(amount="300") # 3.00 USD
preview = r.json()
r.status_code = preview

### Create offers

How to use the client to generate L402 offers

In [ ]:
#| export
@patch
def create_offers(self:Fewsats,
                 offers:List[Dict[str,Any]], # List of offer objects following OfferCreateV0 schema
) -> dict:
    "Create offers for L402 payment server"
    return self._request("POST", "v0/l402/offers", json={"offers": offers})

In [ ]:
test_offers = [{
    "id": "test_offer_2",
    "amount": 1,
    "currency": "usd" ,
    "description": "Test offer",
    "title": "Test Package",
    "payment_methods": ["lightning", "credit_card"]
}]

r = fs.create_offers(test_offers)
l402_offers = r.json()
r.status_code, l402_offers

(200,
 {'offers': [{'id': 'test_offer_2',
    'amount': 1,
    'currency': 'usd',
    'description': 'Test offer',
    'title': 'Test Package',
    'payment_methods': ['lightning', 'credit_card'],
    'type': 'one-off'}],
  'payment_context_token': 'b413ee66-a5b4-4168-88e2-5735166f1984',
  'payment_request_url': 'http://localhost:8000/v0/l402/payment-request',
  'version': '0.2.2'})

### Get Payment Details

Get payment details is a convenience method for buyers to retrieve the payment information like stripe checkout url, lightning invoice etc... It does not need to be used by vendors. We demonstrate it here to showcase how an offer generated by a vendor can be turned into actual payments details.

In [ ]:
#| export
@patch
def get_payment_details(self:Fewsats,
                       payment_request_url:str, # The payment request URL
                       offer_id:str, # The offer ID
                       payment_method:str, # The payment method (lightning, credit_card, ...)
                       payment_context_token:str, # The payment context token
                       ) -> dict:
    """Gets payment details for a specific offer. Use this as buyer when you want to make the payment manually."""
    data = {"offer_id": offer_id, "payment_method": payment_method, "payment_context_token": payment_context_token}
    return httpx.post(payment_request_url, json=data)


In [ ]:
r = fs.get_payment_details(l402_offers["payment_request_url"], l402_offers["offers"][0]["id"], "lightning", l402_offers["payment_context_token"])
payment_details = r.json()
ln_invoice = payment_details["payment_request"]['lightning_invoice']
r.status_code, payment_details

(200,
 {'expires_at': '2025-04-28T16:43:07.985112+00:00',
  'offer_id': 'test_offer_2',
  'payment_request': {'lightning_invoice': 'lnbc100n1p5qlflfpp50zwmzn36u93u7hcnhyjtqgsa65lnrrzfymeetv6uyqrltnktpesqdq523jhxapq2pskx6mpvajscqzpgxqrzpjrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp5kppjx7rg9kq0ugvgn4vkcfa4z34ytcjtylcfxhs3dwl2sr029m8s9qxpqysgqtgunz5j2wkm5845qmvk2m7yz5nnqwtpnhqteawq97wv3m9xneeusjuj7xjp88078th6rtx09ad7zt3cc3l5dd3w9hkakfpkw3vvm0ugqgm2lc4'},
  'version': '0.2.2'})

### Get Payment Status

In [ ]:
#| export
@patch
def get_payment_status(self:Fewsats, 
                       payment_context_token:str, # The payment context token
                       ) -> dict:
    """Gets the status of a submitted payment. 
    Vendors should use this to check if anyone has paid for their offer associated with the token."""
    return self._request("GET", f"v0/l402/payment-status?payment_context_token={payment_context_token}")

In [ ]:

r = fs.get_payment_status(l402_offers["payment_context_token"])
r.status_code, r.json()

(200,
 {'payment_context_token': 'b413ee66-a5b4-4168-88e2-5735166f1984',
  'status': 'pending',
  'offer_id': None,
  'paid_at': None,
  'amount': None,
  'currency': None})

### Webhooks

Use this as a vendor to get notified when someone pays for your offer. Currently only 1 webhook is supported per user.

In [ ]:
#| export
@patch
def add_webhook(self:Fewsats,
                       webhook_url:str,
                       ) -> dict:
    """Add a URL to the list of webhooks that will receive notifications when you receive a payment.
    The webhook will be triggered for every successful payment."""
    return self._request("POST", f"v0/users/webhook/add", json={"webhook_url": webhook_url})

In [ ]:

r = fs.add_webhook("https://example.com")
r, r.json()

(<Response [200 OK]>,
 {'id': 4, 'url': 'https://example.com', 'is_test': False})

To verify and parse a webhook that comes from Fewsats you can use this utility

In [ ]:
#| export

@dataclass
class FewsatsWebhookEvent:
    offer_id: str
    payment_context_token: str
    amount: int
    currency: str
    status: str
    timestamp: str

@patch(cls_method=True)
def verify_webhook(cls:Fewsats,
                   data: bytes,
                   signature: str,
                   webhook_secret: str,
                   ) -> dict:
    """
    Verify and parse a webhook that comes from Fewsats
    Args:
        data: bytes
        signature: str
        webhook_secret: str
    Returns:
        FewsatsWebhookEvent
    """
    if not isinstance(data, bytes):
        raise TypeError(f"'data' should be bytes, got {type(data)}")

    timestamp_str, signature_str = signature.split(",")
    timestamp = timestamp_str.split("=")[1]
    signature_version, signature = signature_str.split("=")

    payload_str = data.decode("utf-8")

    if signature_version != cls.WEBHOOK_VERSION:
        raise ValueError("Unsupported signature version")

    if timestamp.isdigit():
        timestamp = int(timestamp)
    else:
        raise ValueError("Invalid timestamp")

    if timestamp - time.time() > 300:
        raise ValueError("Timestamp is older than 5 minutes")

    signed_payload = f"{timestamp}.{payload_str}"

    # Generate the signature
    expected_signature = hmac.new(webhook_secret.encode(), signed_payload.encode(), hashlib.sha256).hexdigest()

    if signature.lower() != expected_signature.lower():
        raise ValueError("Webhook message hash does not match signature")

    event = json.loads(payload_str)
    return FewsatsWebhookEvent(**event)


In [ ]:
data = bytes(json.dumps({
  "offer_id": "offer-501040",
  "payment_context_token": "a0b1caf3-3e3b-488f-ae9c-18d64f690894",
  "amount": 812319,
  "currency": "USD",
  "status": "failed",
  "timestamp": "2025-04-16T08:51:12Z"
}, sort_keys=True, separators=(',', ':')), "utf-8")
signature = "t=1744793472,v1=89a491b8f3f8e72b75896faa24cb1cfade27bea12bbdfe333759809b8a573ad3"

event = Fewsats.verify_webhook(data, signature, "whsec_bIi4m3by9sJ_KNfY5PFWb2YmAqm2WVAvwq5wGuphayE")
event

FewsatsWebhookEvent(offer_id='offer-501040', payment_context_token='a0b1caf3-3e3b-488f-ae9c-18d64f690894', amount=812319, currency='USD', status='failed', timestamp='2025-04-16T08:51:12Z')

## Pay Methods

### Pay Lightning Invoice

Pay lightning invoice is a low-level method to manually pay for a lightning invoice. 

In [ ]:
#| export

@patch
def pay_lightning(self: Fewsats, 
                  invoice: str, # lightning invoice
                  amount: int, # amount in cents
                  currency: str = "usd", # currency
                  description: str = "" ): # description of the payment 
    "Pay for a lightning invoice directly."
    data = {
        "invoice": invoice,
        "amount": amount,
        "currency": currency,
        "description": description
    }
    return self._request("POST", "v0/l402/purchases/lightning", json=data)

In [ ]:
r = fs.pay_lightning(invoice=ln_invoice,
                     description="fewsats webhook trial", amount=1)
lightning_payment = r.json()
r.status_code, lightning_payment


(200,
 {'id': '5cb91f55-447f-4a6c-981f-902ebcaeb35d',
  'created_at': '2025-04-23T12:35:14.126Z',
  'status': 'success',
  'payment_request_url': '',
  'payment_context_token': '',
  'invoice': 'lnbc100n1p5q3h5ppp54neu5ukgkshem4j6yyx4gdpuscj8psy92f52jhgzxsts6az02twsdq523jhxapq2pskx6mpvajscqzpgxqrzpnrzjqwghf7zxvfkxq5a6sr65g0gdkv768p83mhsnt0msszapamzx2qvuxqqqqz99gpz55yqqqqqqqqqqqqqq9qrzjq25carzepgd4vqsyn44jrk85ezrpju92xyrk9apw4cdjh6yrwt5jgqqqqz99gpz55yqqqqqqqqqqqqqq9qsp52p0m2rgrd8v6peyxm2dqwwyfn32wmls3x74thr890h0rx2nt3h2q9qxpqysgqu6rsscdg76jng554uj7pt7ln7kage37u9t2ut4208suzvu5e4hw5n2ssa4dhmfpgc0mgmywzvya9902sg8re846pn6f4fkl3vhpa80cp7jqjjr',
  'preimage': '92106e710f38997a2c96b53665e3f1bb0616419eb4d28bcf33ae2803f2066b1d',
  'amount': 1,
  'currency': 'usd',
  'payment_method': 'lightning',
  'title': '',
  'description': 'fewsats webhook trial',
  'type': '',
  'is_test': False})

In [ ]:
r = fs.pay_lightning(invoice='lnbc100n1p5qlv5dpp5r4jjrenyvlndnr59nx2fgk8azyeaaxzpeuy7putgm07wfnvt398qdqqcqzzsxqrrsssp54897npsmum0av3qqj6jyvl3sks4hlc67ugdyr8x85hla79u262uq9qxpqysgq3m9p2a8j7fy2pxeg5xuhyw4zwwp6md5egkez28afffmsch2l7clhw5922mxlr52gp48w0pdvnngytgj08kn4z8uv25pq7fc0mxhrs6qphe3c0k',
                     description="fewsats webhook trial", amount=1)
r.status_code, r.text

### Paying L402 Offers

The pay method pays for a specific offer. The user is not required to fetch the payment details beforehand. It is asynchronous and returns the `payment_id` and `status`. Using the `payment_id` we can check the status of the payment.


There are two versions of the pay offer. One accepts the L402 offers as a string, which is more convenient for text-based AI agents like LLMs. The other accepts either the `L402Offers` custom class - supported by some libraries like [Claudette](https://claudette.answer.ai/core.html#tool) - or a simple `Dict`.

In [ ]:
#| export

class Offer(BasicRepr):
    "Represents a single L402 offer"
    def __init__(self, 
                 id: str,
                 amount: int,
                 currency: str,
                 description: str,
                 title: str,
                 payment_methods: List[str] = None,
                 type: str = "one-off"): 
        store_attr()

    def __repr__(self):
        return f"Offer: {self.title}\nID: {self.id}\nAmount: {self.amount/100} {self.currency}\nDescription: {self.description}"
    
    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> 'Offer':
        "Create an Offer from a dictionary"
        return cls(**d)

class L402Offers(BasicRepr):
    "Represents the complete L402 offers schema"
    def __init__(self, 
                 offers: List[Offer],
                 payment_context_token: str,
                 payment_request_url: str,
                 version: str): 
        store_attr()
    
    def __repr__(self):
        offers_str = "\n".join([f"- {o.title} ({o.amount/100} {o.currency})" for o in self.offers])
        return f"L402 Offers:\n{offers_str}\nPayment URL: {self.payment_request_url}\nContext Token: {self.payment_context_token}"
    
    def as_dict(self) -> Dict[str, Any]:
        "Convert to dictionary format for API usage"
        return {
            'offers': [vars(o) for o in self.offers],
            'payment_context_token': self.payment_context_token,
            'payment_request_url': self.payment_request_url,
            'version': self.version
        }
    
    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> 'L402Offers':
        "Create an L402Offers object from a dictionary"
        offers = [Offer.from_dict(o) for o in d['offers']]
        return cls(
            offers=offers,
            payment_context_token=d['payment_context_token'],
            payment_request_url=d['payment_request_url'],
            version=d['version']
        )


In [ ]:
l402 = L402Offers.from_dict(l402_offers)
l402

L402 Offers:
- Test Package (0.01 usd)
Payment URL: http://localhost:8000/v0/l402/payment-request
Context Token: b413ee66-a5b4-4168-88e2-5735166f1984

In [ ]:
# we make sure that invalid offers fail
invalid_json = {
    'offers': [{'id': 'test_offer_2', 'amount': 1}],  # Missing fields
    'payment_context_token': '60a8e027-8b8b-4ccf-b2b9-380ed0930283'
    # Missing payment_request_url
}
test_fail(lambda: Offer(id="test", amount=1), 
          contains="missing 3 required positional arguments: 'currency', 'description', and 'title'")


### Pay L402 Offer


In [ ]:
#| export 

@patch
def pay_offer(self:Fewsats,
        offer_id : str, # the offer id to pay for
        l402_offer: L402Offers, # a dictionary containing L402 offers
) -> dict: # payment status response
    """Pays an offer_id from the l402_offers. 
    The l402_offer parameter must be a dictionary with this structure:
    {
        'offers': [
            {
                'id': 'test_offer_2',  # String identifier for the offer
                'amount': 1,                 # USD cents
                'currency': 'usd',           # Currency code
                'description': 'Test offer', # Text description
                'title': 'Test Package'      # Title of the package
            }
        ],
        'payment_context_token': 'token',  # Payment context token
        'payment_request_url': 'https://api.fewsats.com/v0/l402/payment-request',  # Payment URL
        'version': '0.2.2'  # API version
    }
    Returns payment status response"""
    if isinstance(l402_offer, dict): l402_offer = L402Offers.from_dict(l402_offer)
    offer_dict = l402_offer.as_dict()
    data = {"offer_id": offer_id, **offer_dict}
    return self._request("POST", "v0/l402/purchases/from-offer", json=data)


In [ ]:
offer_id = l402.offers[0].id
r = fs.pay_offer(offer_id, l402)
payment_response = r.json()
r.status_code, payment_resp2onse

(200,
 {'id': 'd1e8abcd-a3a5-425b-b6a4-073559f772b6',
  'created_at': '2025-04-23T12:35:17.528Z',
  'status': 'success',
  'payment_method': 'lightning'})

### L402 Pay Offer with JSON string

This alternative method accepts a JSON string containing L402 offers. It should be used by systems that do not support custom classes in tool calling.

In [ ]:
#| export

@patch
def pay_offer_str(self:Fewsats,
        offer_id : str, # the offer id to pay for
        l402_offer: str, # JSON string containing L402 offers
) -> dict: # payment status response
    """Pays an offer_id from the l402_offers.

    The l402_offer parameter must be a JSON string with this structure:
    {
        'offers': [
            {
                'id': 'test_offer_2',  # String identifier for the offer
                'amount': 1,                 # Numeric cost value
                'currency': 'usd',           # Currency code
                'description': 'Test offer', # Text description
                'title': 'Test Package'      # Title of the package
            }
        ],
        'payment_context_token': '60a8e027-8b8b-4ccf-b2b9-380ed0930283',  # Payment context token
        'payment_request_url': 'https://api.fewsats.com/v0/l402/payment-request',  # Payment URL
        'version': '0.2.2'  # API version
    }

    Returns payment status response"""
    # Parse JSON string to dictionary
    try:
        offer_data = json.loads(l402_offer)
        L402Offers.from_dict(offer_data) # we don't care about the return value, just validating the json input
    except json.JSONDecodeError:
        raise ValueError("Invalid JSON string provided for l402_offer")
    
    # Create payload with offer_id
    data = {"offer_id": offer_id, **offer_data}
    
    return self._request("POST", "v0/l402/purchases/from-offer", timeout=20, json=data)

In [ ]:
r = fs.pay_offer_str(l402_offers["offers"][0]["id"], json.dumps(l402_offers))
r.status_code, r.json()

(400,
 {'detail': "Invalid payment request received. Payment context token 'b413ee66-a5b4-4168-88e2-5735166f1984' already used"})


### Pay Link

In [ ]:
#| export

@patch
def pay_link(self:Fewsats,
        url: str, # URL to purchase from
        description: str, # Description of the purchase
        price: int, # Price in USD cents
        payment_method: str = "credit_card", # Payment method (credit_card, lightning, etc.)
        title: str = "Purchase from URL" # Title of the purchase
) -> dict: # payment status response
    """Creates a purchase record for an external URL.
    
    Args:
        url: The URL to purchase from
        description: Description of the purchase
        price: Price in USD cents
        payment_method: Payment method to use (default: credit_card)
        title: Title of the purchase (default: "Purchase from URL")
        
    Returns:
        Payment status response containing information about the purchase
    """
    data = {
        "url": url,
        "description": description,
        "price": price,
        "payment_method": payment_method,
        "title": title
    }
    return self._request("POST", "v0/l402/purchases/from-link", json=data)

In [ ]:
url = "https://www.amazon.com/Matter-Culture-Iain-M-Banks/dp/0316005371"
r = fs.pay_link(url, "Matter (Culture)", 629)
r.status_code, r.text

(200,
 '{"id": "5135d358-bb18-45f4-a20a-9cb8961cbf35", "created_at": "2025-04-23T12:35:17.917Z", "status": "pending", "payment_method": "credit_card"}')

### Payment Info

As a buyer, we can check the status of a payment as follows:

In [ ]:
#| export
@patch
def payment_info(self:Fewsats,
                  pid:str): # purchase id
    "Retrieve the details of a payment."
    return self._request("GET", f"v0/l402/outgoing-payments/{pid}")

In [ ]:
r = fs.payment_info(payment_response['id'])
r.status_code, r.json()

(200,
 {'id': 'd1e8abcd-a3a5-425b-b6a4-073559f772b6',
  'created_at': '2025-04-23T12:35:17.528Z',
  'status': 'success',
  'payment_request_url': '',
  'payment_context_token': 'b413ee66-a5b4-4168-88e2-5735166f1984',
  'invoice': '',
  'preimage': '',
  'amount': 1,
  'currency': 'usd',
  'payment_method': 'lightning',
  'title': 'Test Package',
  'description': 'Test offer',
  'type': 'one-off',
  'is_test': False})

### Pay X402 Offer

In [ ]:
#| export

@patch
def pay_x402_offer(self:Fewsats,
                               payload:Dict[str, Any], # The x402 offer payload
                               chain:str = "base", # Blockchain chain to use
                              ) -> dict:
    """Creates a payment from an x402 offer and returns a payment header to access the resource.
    
    Args:
        payload: The x402 offer payload containing accepts, error, and x402Version
        chain: Blockchain chain to use (default: "base")
        
    Returns:
        Dictionary containing payment_header to use for subsequent requests
    """
    data = {
        "chain": chain,
        "payload": payload
    }
    return self._request("POST", "v0/x402/purchases/from-offer", json=data)

In [ ]:
x402_offer = {
  "accepts": [
    {
      "scheme": "exact",
      "network": "base-sepolia",
      "maxAmountRequired": "1",
      "resource": "https://proxy402.com/KQzW9kfi3Z",
      "description": "Payment for GET https://proxy402.com/KQzW9kfi3Z",
      "mimeType": "",
      "payTo": "0xddb24Bd8A6Cb0f2d3eaBF7a828C0b4364668B963",
      "maxTimeoutSeconds": 60,
      "asset": "0x036CbD53842c5426634e7929541eC2318f3dCF7e",
      "extra": {
        "name": "USDC",
        "version": "2"
      }
    }
  ],
  "error": "X-PAYMENT header is required",
  "x402Version": 1
}

r = fs.pay_x402_offer(x402_offer, chain="base-sepolia")
r.status_code, r.json()

(200,
 {'payment_header': 'eyJ4NDAyVmVyc2lvbiI6IDEsICJzY2hlbWUiOiAiZXhhY3QiLCAibmV0d29yayI6ICJiYXNlLXNlcG9saWEiLCAicGF5bG9hZCI6IHsic2lnbmF0dXJlIjogIjB4MTU4NWExZjcyZjk3ZDU3ZmM3NjliZmRlNjZjNjQ0NzNlMzAxYzYwYjg0YjJiZTk5NDFiN2ZmMTRkMTc4MzIyOTQzOGRlNGI2ZGU5YjdlMzlkMDZiMmEwY2I0YTA0ZjQ1MDE5ZTExYmJjOWI5OWEyZGQ3OTQ0YTNmYzgwOTdiNjExYyIsICJhdXRob3JpemF0aW9uIjogeyJmcm9tIjogIjB4MjNiODExMDllODFGREZFOGU0MjEzNUU4M0MzMWIxMUFhN0Q5OTlBNCIsICJ0byI6ICIweGRkYjI0QmQ4QTZDYjBmMmQzZWFCRjdhODI4QzBiNDM2NDY2OEI5NjMiLCAidmFsdWUiOiAiMSIsICJ2YWxpZEFmdGVyIjogIjE3NDc0MDAyNDEiLCAidmFsaWRCZWZvcmUiOiAiMTc0NzQwMDM2MSIsICJub25jZSI6ICIweGI0ODI1M2NiMTJlMzMyNDE3N2ZiODgwYzU4OThkYzU5NmVlOGM2OWRkNGYyN2JjZTI0MGUwNzg1MzdiMTJjMzgifX19'})

### Pay X402 Link

In [ ]:
#| export

@patch
def pay_x402_link(self:Fewsats,
                              url:str, # URL to purchase from
                              method:str = "GET", # HTTP method to use
                              body:Dict[str, Any] = None, # Optional request body
                              headers:Dict[str, str] = None, # Optional request headers
                              chain:str = "base", # Blockchain chain to use
                             ) -> dict:
    """Creates a purchase from an external URL that requires x402 payment.
    
    Args:
        url: The URL to purchase from
        method: HTTP method to use (default: "GET")
        body: Optional request body
        headers: Optional request headers
        chain: Blockchain chain to use (default: "base")
        
    Returns:
        The response from the target URL after successful payment
    """
    data = {
        "url": url,
        "method": method,
        "chain": chain
    }
    
    if body is not None:
        data["body"] = body
        
    if headers is not None:
        data["headers"] = headers
        
    return self._request("POST", "v0/x402/purchases/from-link", json=data)

In [ ]:
x402_link = "https://proxy402.com/KQzW9kfi3Z"
r = fs.pay_x402_link(x402_link, chain="base-sepolia")
r.status_code, r.json()

(400, {'detail': 'Redirecting...\n'})

## As tools

In [ ]:
#| export

def get_response(r): return r.status_code, r.text

def wrap_with_response(method):
    """Wraps a method to return (status_code, text) instead of a Response object"""
    @wraps(method)  # This preserves name, docstring, signature
    def wrapped(*args, **kwargs):
        response = method(*args, **kwargs)
        return get_response(response)
    return wrapped

@patch
def as_tools(self:Fewsats):
    "Return list of available tools for AI agents"
    methods = [self.me, self.balance, self.payment_methods, self.pay_offer_str, self.payment_info]
    return [wrap_with_response(m) for m in methods]

In [ ]:
fs.as_tools()

[<function __main__.Fewsats.me()>,
 <function __main__.Fewsats.balance()>,
 <function __main__.Fewsats.payment_methods() -> List[Dict[str, Any]]>,
 <function __main__.Fewsats.pay_offer_str(offer_id: str, l402_offer: str) -> dict>,
 <function __main__.Fewsats.payment_info(pid: str)>]

Both the preview and purchase methods automatically use the default payment method if a charge is needed. This client provides a straightforward way to interact with the Fewsats API, making it easy for developers to integrate Fewsats functionality into their applications.

## Agent Demo

We will use [Claudette](https://claudette.answer.ai/) to demonstrate how to pay for content using the Fewsats API.

In [ ]:
from claudette import Chat, models

In [ ]:
model = models[1]
model

'claude-3-5-sonnet-20240620'

In [ ]:
fs.balance().json()

[{'id': 1, 'balance': 4395, 'currency': 'usd'}]

In [ ]:
# let's create a payable offer
test_offers = [{
    "id": "test_offer_2",
    "amount": 1,
    "currency": "usd" ,
    "description": "Test offer",
    "title": "Test Package",
    "payment_methods": ["lightning", "credit_card"]
}]

l402_offers = fs.create_offers(test_offers).json()

In [ ]:
chat = Chat(model, sp='You are a helpful assistant that can pay offers.', tools=fs.as_tools())
pr = f"Could you pay the cheapest offer in {l402_offers}?"
r = chat.toolloop(pr, trace_func=print)
r

Message(id='msg_01FeTBy9s6R7N3Mup7ADYhek', content=[TextBlock(text="Certainly! I'll analyze the offer information you provided and pay for the cheapest offer. In this case, there's only one offer available, so that will be the one we'll pay for. Let's proceed with the payment using the `pay_offer_str` function.", type='text'), ToolUseBlock(id='toolu_01WwGc8fUjhugjme9uk2RUCh', input={'offer_id': 'test_offer_2', 'l402_offer': '{"offers": [{"id": "test_offer_2", "amount": 1, "currency": "usd", "description": "Test offer", "title": "Test Package", "payment_methods": ["lightning", "credit_card"], "type": "one-off"}], "payment_context_token": "6985d210-8205-4c80-b0e0-821ccfd543ef", "payment_request_url": "http://localhost:8000/v0/l402/payment-request", "version": "0.2.2"}'}, name='pay_offer_str', type='tool_use')], model='claude-3-5-sonnet-20240620', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=In: 1032; Out: 268; Cache create: 0; Cache read: 0; Total: 

Great news! The payment for the offer has been successfully processed. Here's a summary of the transaction:

1. Offer ID: test_offer_2
2. Amount: 1 USD cent (0.01 USD)
3. Title: Test Package
4. Description: Test offer
5. Payment ID: c0af1684-3889-4974-92e2-aae21b1b46ea
6. Payment Status: Success
7. Payment Method: Lightning
8. Transaction Date: April 23, 2025, at 12:45:22 UTC

The payment was completed successfully using the Lightning network. Is there anything else you would like to know about this transaction or any other assistance you need?

<details>

- id: `msg_01BCSCYLy3cjZEQCziHHG9Dy`
- content: `[{'text': "Great news! The payment for the offer has been successfully processed. Here's a summary of the transaction:\n\n1. Offer ID: test_offer_2\n2. Amount: 1 USD cent (0.01 USD)\n3. Title: Test Package\n4. Description: Test offer\n5. Payment ID: c0af1684-3889-4974-92e2-aae21b1b46ea\n6. Payment Status: Success\n7. Payment Method: Lightning\n8. Transaction Date: April 23, 2025, at 12:45:22 UTC\n\nThe payment was completed successfully using the Lightning network. Is there anything else you would like to know about this transaction or any other assistance you need?", 'type': 'text'}]`
- model: `claude-3-5-sonnet-20240620`
- role: `assistant`
- stop_reason: `end_turn`
- stop_sequence: `None`
- type: `message`
- usage: `{'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 1380, 'output_tokens': 165}`

</details>

In [ ]:
chat.h

[{'role': 'user',
  'content': [{'type': 'text',
    'text': "Could you pay the cheapest offer in {'offers': [{'id': 'test_offer_2', 'amount': 1, 'currency': 'usd', 'description': 'Test offer', 'title': 'Test Package', 'payment_methods': ['lightning', 'credit_card'], 'type': 'one-off'}], 'payment_context_token': '6985d210-8205-4c80-b0e0-821ccfd543ef', 'payment_request_url': 'http://localhost:8000/v0/l402/payment-request', 'version': '0.2.2'}?"}]},
 {'role': 'assistant',
  'content': [TextBlock(text="Certainly! I'll analyze the offer information you provided and pay for the cheapest offer. In this case, there's only one offer available, so that will be the one we'll pay for. Let's proceed with the payment using the `pay_offer_str` function.", type='text'),
   ToolUseBlock(id='toolu_01WwGc8fUjhugjme9uk2RUCh', input={'offer_id': 'test_offer_2', 'l402_offer': '{"offers": [{"id": "test_offer_2", "amount": 1, "currency": "usd", "description": "Test offer", "title": "Test Package", "payment_m

In [ ]:
#|hide
from nbdev.doclinks import nbdev_export
nbdev_export()